# Lot 42 — Hard-position MCTS65536 extension

Exécute MCTS65536 uniquement sur les positions difficiles Lot41B. Ne relance pas les 256 positions.

In [ ]:
from pathlib import Path
import os, subprocess, sys, time

REPO_URL = 'https://github.com/GlennEriss/songo-fish.git'
REPO_DIR = Path('/content/songo-fish')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[train,perf,dev]'], check=True)
os.environ['PYTHONPATH'] = 'packages:apps/trainer/scripts'
print('Projet installé depuis :', Path.cwd())

In [ ]:
from google.colab import drive, files
DRIVE_AVAILABLE = False
try:
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
except Exception as exc:
    print('Drive indisponible:', exc)

DRIVE_ROOT = Path('/content/drive/MyDrive/songo-ai')
LOT41 = DRIVE_ROOT / 'experiments/lot41_deep_mcts_convergence' if DRIVE_AVAILABLE else Path('/content/lot41_deep_mcts_convergence')
LOT41B = DRIVE_ROOT / 'experiments/lot41b_deep_mcts_strategic_convergence' if DRIVE_AVAILABLE else Path('/content/lot41b_deep_mcts_strategic_convergence')
OUTPUT = DRIVE_ROOT / 'experiments/lot42_hard_position_65536' if DRIVE_AVAILABLE else Path('/content/lot42_hard_position_65536')
EXPORT = DRIVE_ROOT / 'exports/lot42_results.tar.gz' if DRIVE_AVAILABLE else Path('/content/lot42_results.tar.gz')
OUTPUT.mkdir(parents=True, exist_ok=True)
EXPORT.parent.mkdir(parents=True, exist_ok=True)
print('LOT41 =', LOT41)
print('LOT41B =', LOT41B)
print('OUTPUT =', OUTPUT)

In [ ]:
def supervised_run(cmd, label):
    log = Path('/content') / f'{label}.log'
    with log.open('wb') as stream:
        p = subprocess.Popen(cmd, stdout=stream, stderr=subprocess.STDOUT)
    off = 0
    started = time.time()
    while p.poll() is None:
        time.sleep(10)
        data = log.read_bytes()[off:]
        off += len(data)
        if data:
            print(data.decode(errors='replace'), end='', flush=True)
        print(f'[{label}] actif depuis {int(time.time()-started)}s', flush=True)
    data = log.read_bytes()[off:]
    if data:
        print(data.decode(errors='replace'), end='', flush=True)
    print(f'[{label}] code de sortie={p.returncode}; journal={log}', flush=True)
    if p.returncode:
        raise RuntimeError(f'{label} a échoué avec le code {p.returncode}; consulter {log}')

RUN = [sys.executable, 'apps/trainer/scripts/run_srn_lot42.py']

In [ ]:
required = [
    LOT41 / 'search/budget_32768.json',
    LOT41B / 'decision.json',
    LOT41B / 'hard_positions.json',
]
for path in required:
    print(path, 'OK' if path.exists() else 'MISSING')
assert all(path.exists() for path in required), 'Lot41/Lot41B artifacts missing'

In [ ]:
supervised_run(RUN + ['--stage','prepare','--device','cuda','--lot41',str(LOT41),'--lot41b',str(LOT41B),'--output',str(OUTPUT),'--concurrency','4'], 'lot42_prepare')

In [ ]:
# Petit pilote exact MCTS65536 sur les premières hard positions. Résultat réutilisé par run.
supervised_run(RUN + ['--stage','pilot','--device','cuda','--lot41',str(LOT41),'--lot41b',str(LOT41B),'--output',str(OUTPUT),'--pilot-count','2','--concurrency','4'], 'lot42_pilot')

In [ ]:
# Ajuster --concurrency si la RAM Colab est limitée. 4 est prudent; 8 peut être tenté en haute RAM.
supervised_run(RUN + ['--stage','run','--device','cuda','--lot41',str(LOT41),'--lot41b',str(LOT41B),'--output',str(OUTPUT),'--concurrency','4'], 'lot42_run')

In [ ]:
supervised_run(RUN + ['--stage','finalize','--lot41',str(LOT41),'--lot41b',str(LOT41B),'--output',str(OUTPUT)], 'lot42_finalize')
supervised_run(RUN + ['--stage','export','--output',str(OUTPUT),'--bundle',str(EXPORT)], 'lot42_export')
print('decision.json:', OUTPUT / 'decision.json')
print('export:', EXPORT)
if not DRIVE_AVAILABLE:
    files.download(str(EXPORT))
    files.download(str(Path(str(EXPORT) + '.sha256')))